# Лабораторная работа 1 — рабочий ноутбук

**ФИО:** Гончар Александр Сергеевич
**Файлы данных:** `data/clients.csv`, `data/transactions.csv`

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RAW_CLIENTS = "data/clients.csv"
RAW_TXN = "data/transactions.csv"

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

Пробуем прочитать
файл с разными параметрами. Неверная кодировка дает `UnicodeDecodeError`,
неверный разделитель дает один столбец вместо нескольких.

In [2]:
for path in [RAW_CLIENTS, RAW_TXN]:
    for enc in ["utf-8", "cp1251"]:
        for sep in [",", ";"]:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep)
                print(path, enc, repr(sep), "- столбцов:", df.shape[1])
            except UnicodeDecodeError:
                print(path, enc, repr(sep), "- ошибка кодировки")

data/clients.csv utf-8 ',' - столбцов: 16
data/clients.csv utf-8 ';' - столбцов: 1
data/clients.csv cp1251 ',' - ошибка кодировки
data/clients.csv cp1251 ';' - ошибка кодировки
data/transactions.csv utf-8 ',' - ошибка кодировки
data/transactions.csv utf-8 ';' - ошибка кодировки


data/transactions.csv cp1251 ',' - столбцов: 1


data/transactions.csv cp1251 ';' - столбцов: 9


`clients.csv` кодировка UTF-8, разделитель ',' (16 столбцов). `transactions.csv` кодировка cp1251,
разделитель ';' (9 столбцов). Но то, что файл открылся в cp1251, еще не значит, что кодировка верная так как
в cp1251 откроется почти любой файл, просто вместо букв может получиться мусор.
Поэтому дальше будем смотреть на сам текст.

Смотрим первые строки transactions и проверяем, что русские слова читаются.

In [3]:
pd.read_csv(RAW_TXN, encoding="cp1251", sep=";", nrows=5)

,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990102,C003921,2023-11-24 09:00:10,"1018,07",RUB,Кафе/рестораны,5812,Карта,успешно
1,990012,C002226,2023-11-25 17:53:53,"454,49",RUB,Продукты,5411,Карта,успешно
2,990027,C000544,2023-11-27 13:30:37,"1194,49",RUB,Продукты,5411,Онлайн-банк,успешно
3,990169,C002702,2023-11-27,"755,99",RUB,Онлайн-сервисы,5815,Карта,успешно
4,990082,C001351,2023-12-12,"1911,83",RUB,Продукты,5411,Карта,успешно


Категории, каналы и статусы читаются как нормальные русские слова,
значит кодировка cp1251 выбрана верно.
Также в amount дробная часть отделена запятой.

Читаем `transactions.csv` без указания десятичного разделителя и смотрим тип amount.

In [4]:
txn_try = pd.read_csv(RAW_TXN, encoding="cp1251", sep=";")
print(txn_try["amount"].dtype, txn_try["amount"].head(3).tolist())

str ['1018,07', '454,49', '1194,49']


Видим, что amount прочитался как текст, так как pandas по умолчанию ждет точку, а в файле запятая. Поэтому указываем `decimal=","`.

In [7]:
txn_try = pd.read_csv(RAW_TXN, encoding="cp1251", sep=";", decimal=",")
print(txn_try["amount"].dtype, txn_try["amount"].head(5).tolist())

float64 [1018.07, 454.49, 1194.49, 755.99, 1911.83]


С `decimal=","` amount стал числом. pandas делает столбец числовым только если смог перевести каждое значение, так что все суммы прочитались правильно.

## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

### 2.2. Дубли

### 2.3. Соответствие данных описанию полей из задания

### 2.4. Связь между таблицами

**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

### 3.2. Категориальные признаки

### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [6]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [7]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")